# 🎬 Georgian Cartoon Dubbing — Colab (GPU)

Runs the whole pipeline on a free cloud GPU. **Before you start:**
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Add three keys in the **🔑 Secrets** panel (left sidebar), toggling *Notebook access* on for each:
   `ELEVENLABS_API_KEY`, `ANTHROPIC_API_KEY`, `HF_TOKEN`.
3. Put your episode in Google Drive, e.g. `MyDrive/dubbing/episode.mp4`.
4. Accept the pyannote licenses once (logged in): [diarization-3.1](https://hf.co/pyannote/speaker-diarization-3.1) and [segmentation-3.0](https://hf.co/pyannote/segmentation-3.0).

Then run the cells in order.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

## 2. Get the pipeline code
Set `REPO_URL` to your pushed `dub-pipeline` repo. (Private repo? Use `https://<TOKEN>@github.com/you/dub-pipeline.git`.)

*No GitHub? See the commented zip-from-Drive fallback below.*

In [ ]:
REPO_URL = 'https://github.com/YOUR_USERNAME/dub-pipeline.git'  # <-- set this
!git clone -q $REPO_URL
%cd dub-pipeline

# --- Fallback: no GitHub. Put dub-pipeline.zip in MyDrive/dubbing/ and use: ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/dubbing/dub-pipeline.zip . && unzip -q dub-pipeline.zip
# %cd dub-pipeline

## 3. Install dependencies
Takes a few minutes. If a later cell reports CUDA missing, re-run this one.

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install -r requirements.txt
print('deps installed')

## 4. Load your API keys from Colab Secrets

In [ ]:
from google.colab import userdata
keys = ['ELEVENLABS_API_KEY', 'ANTHROPIC_API_KEY', 'HF_TOKEN']
with open('.env', 'w') as f:
    for k in keys:
        v = userdata.get(k)
        assert v, f'Missing secret: {k} (add it in the 🔑 panel)'
        f.write(f'{k}={v}\n')
print('keys written to .env')

## 5. Mount Drive & point at your episode

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
VIDEO = '/content/drive/MyDrive/dubbing/episode.mp4'  # <-- your file
import os; assert os.path.exists(VIDEO), f'Not found: {VIDEO}'
print('video ok:', VIDEO)

## 6. Config & character voices
Run this once now (with placeholders). After **Pass 1** reveals the speaker labels, come back, fill in a `voice_id` per character, and re-run this cell.

In [ ]:
import yaml

# After Pass 1, map each detected SPEAKER_xx to an ElevenLabs voice.
VOICES = {
    'SPEAKER_00': {'voice_id': 'REPLACE_ME', 'model_id': 'eleven_multilingual_v2'},
    'SPEAKER_01': {'voice_id': 'REPLACE_ME', 'model_id': 'eleven_flash_v2_5'},
}

config = {
    'source_lang': 'nl', 'target_lang': 'ka',
    'whisper_model': 'large-v3',
    'translate_model': 'claude-sonnet-5',
    'chars_per_second': 15,
    'default_tts_model': 'eleven_flash_v2_5',
    'default_voice_id': 'REPLACE_ME',   # fallback for unmapped speakers
    'output_format': 'mp3_44100_128',
    'voices': VOICES,
    'background_gain_db': -3.0,
    'max_speedup': 1.30, 'min_slowdown': 0.75,
}
yaml.safe_dump(config, open('config.yaml', 'w'), allow_unicode=True, sort_keys=False)
print(open('config.yaml').read())

## 7. Pass 1 — extract, separate, transcribe & detect speakers
The slow GPU stages. When done, note which `SPEAKER_xx` is which character, then update the voices in cell 6 and re-run cell 6.

In [ ]:
!python dub.py run "$VIDEO" --stop-after transcribe

import glob, json
m = json.load(open(glob.glob('work/*/manifest.json')[0], encoding='utf-8'))
print('\n--- detected lines (first 20) ---')
for s in m['segments'][:20]:
    print(f"[{s['speaker']}] {s['start']:.1f}s: {s['text_src']}")

## 8. Pass 2 — translate, preview cost, dub & splice
Make sure you re-ran cell 6 with real voice IDs first. This translates, prints the spend estimate, then synthesizes and rebuilds the video.

In [ ]:
!python dub.py run "$VIDEO" --start-at translate --stop-after translate
!python dub.py estimate "$VIDEO"

In [ ]:
# Synthesize + assemble + mux. --yes skips the interactive prompt (see estimate above).
!python dub.py run "$VIDEO" --start-at tts --yes

# Save the dubbed video back to Drive.
!cp work/*/*.ka.mp4 /content/drive/MyDrive/dubbing/
print('done — check MyDrive/dubbing/ for the *.ka.mp4')